In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

PROCESSED_DIR = Path(r"C:\Projet_filrouge\oncobio_decision_analytics\01_Data\Processed")

# =========================
# 1. Chargement
# =========================
patients_master = pd.read_csv(PROCESSED_DIR / "patients_master.csv")
biomarkers = pd.read_csv(PROCESSED_DIR / "biomarker_reference_clean.csv")
kpi_summary = pd.read_csv(PROCESSED_DIR / "kpi_summary.csv")
patient_biomarker_bridge = pd.read_csv(PROCESSED_DIR / "patient_biomarker_bridge.csv")

# =========================
# 2. Nettoyage patients_master
# =========================
patients_master.columns = patients_master.columns.str.strip()
patients_master["patient_id"] = pd.to_numeric(patients_master["patient_id"], errors="coerce")
patients_master["age"] = pd.to_numeric(patients_master["age"], errors="coerce")
patients_master["survival_months"] = pd.to_numeric(patients_master["survival_months"], errors="coerce")
patients_master["event"] = pd.to_numeric(patients_master["event"], errors="coerce")
patients_master["ecog"] = pd.to_numeric(patients_master["ecog"], errors="coerce")

patients_master["sex"] = patients_master["sex"].astype(str).str.strip()
patients_master["cancer_type"] = patients_master["cancer_type"].astype(str).str.strip()
patients_master["risk_group"] = patients_master["risk_group"].astype(str).str.strip()
patients_master["stage"] = patients_master["stage"].astype(str).str.strip()

patients_master = patients_master.dropna(subset=["patient_id"]).copy()
patients_master["patient_id"] = patients_master["patient_id"].astype(int)

# event doit être entier
patients_master = patients_master.dropna(subset=["event"]).copy()
patients_master["event"] = patients_master["event"].astype(int)

# =========================
# 3. Nettoyage biomarkers
# =========================
biomarkers.columns = biomarkers.columns.str.strip()
biomarkers["biomarker_id"] = pd.to_numeric(biomarkers["biomarker_id"], errors="coerce")
biomarkers["gene_symbol"] = biomarkers["gene_symbol"].astype(str).str.strip()
biomarkers["gene_id"] = biomarkers["gene_id"].astype(str).str.strip()
biomarkers["official_name"] = biomarkers["official_name"].astype(str).str.strip()
biomarkers["description"] = biomarkers["description"].astype(str).str.strip()

biomarkers = biomarkers.dropna(subset=["biomarker_id"]).copy()
biomarkers["biomarker_id"] = biomarkers["biomarker_id"].astype(int)

# =========================
# 4. Nettoyage kpi_summary
# =========================
kpi_summary.columns = kpi_summary.columns.str.strip()
kpi_summary["cancer_type"] = kpi_summary["cancer_type"].astype(str).str.strip()

for col in ["total_patients", "mean_age", "death_rate", "mean_survival", "median_survival"]:
    kpi_summary[col] = pd.to_numeric(kpi_summary[col], errors="coerce")

kpi_summary = kpi_summary.dropna(subset=["cancer_type"]).copy()
kpi_summary["total_patients"] = kpi_summary["total_patients"].fillna(0).astype(int)

# =========================
# 5. Nettoyage patient_biomarker_bridge
# =========================
patient_biomarker_bridge.columns = patient_biomarker_bridge.columns.str.strip()

patient_biomarker_bridge["patient_id"] = pd.to_numeric(
    patient_biomarker_bridge["patient_id"], errors="coerce"
)
patient_biomarker_bridge["biomarker_id"] = pd.to_numeric(
    patient_biomarker_bridge["biomarker_id"], errors="coerce"
)

patient_biomarker_bridge["association_source"] = (
    patient_biomarker_bridge["association_source"]
    .astype(str)
    .str.strip()
)

patient_biomarker_bridge = patient_biomarker_bridge.dropna(
    subset=["patient_id", "biomarker_id"]
).copy()

patient_biomarker_bridge["patient_id"] = patient_biomarker_bridge["patient_id"].astype(int)
patient_biomarker_bridge["biomarker_id"] = patient_biomarker_bridge["biomarker_id"].astype(int)

# =========================
# 6. Création des tables SQL
# =========================

# 1) dim_patients
dim_patients = patients_master[[
    "patient_id", "age", "sex", "cancer_type", "risk_group"
]].copy()

# Nettoyage final
dim_patients["age"] = pd.to_numeric(dim_patients["age"], errors="coerce")
dim_patients["sex"] = dim_patients["sex"].replace({"nan": np.nan})
dim_patients["cancer_type"] = dim_patients["cancer_type"].replace({"nan": np.nan})
dim_patients["risk_group"] = dim_patients["risk_group"].replace({"nan": np.nan})

# 2) fact_clinical_outcomes
fact_clinical_outcomes = patients_master[[
    "patient_id", "survival_months", "event", "stage", "ecog"
]].copy()

fact_clinical_outcomes["survival_months"] = pd.to_numeric(
    fact_clinical_outcomes["survival_months"], errors="coerce"
)
fact_clinical_outcomes["event"] = pd.to_numeric(
    fact_clinical_outcomes["event"], errors="coerce"
)
fact_clinical_outcomes["ecog"] = pd.to_numeric(
    fact_clinical_outcomes["ecog"], errors="coerce"
)

fact_clinical_outcomes = fact_clinical_outcomes.dropna(
    subset=["patient_id", "survival_months", "event"]
).copy()

fact_clinical_outcomes["patient_id"] = fact_clinical_outcomes["patient_id"].astype(int)
fact_clinical_outcomes["event"] = fact_clinical_outcomes["event"].astype(int)

# 3) dim_biomarkers
dim_biomarkers = biomarkers[[
    "biomarker_id", "gene_symbol", "gene_id", "official_name", "description"
]].copy()

# 4) fact_patient_biomarkers
fact_patient_biomarkers = patient_biomarker_bridge[[
    "patient_id", "biomarker_id", "association_source"
]].copy()

fact_patient_biomarkers["patient_id"] = fact_patient_biomarkers["patient_id"].astype(int)
fact_patient_biomarkers["biomarker_id"] = fact_patient_biomarkers["biomarker_id"].astype(int)

# 5) fact_kpis
fact_kpis = kpi_summary[[
    "cancer_type", "total_patients", "mean_age", "death_rate", "mean_survival", "median_survival"
]].copy()

fact_kpis["total_patients"] = pd.to_numeric(fact_kpis["total_patients"], errors="coerce").fillna(0).astype(int)
fact_kpis["mean_age"] = pd.to_numeric(fact_kpis["mean_age"], errors="coerce")
fact_kpis["death_rate"] = pd.to_numeric(fact_kpis["death_rate"], errors="coerce")
fact_kpis["mean_survival"] = pd.to_numeric(fact_kpis["mean_survival"], errors="coerce")
fact_kpis["median_survival"] = pd.to_numeric(fact_kpis["median_survival"], errors="coerce")

# =========================
# 7. Export CSV propres pour PostgreSQL
# =========================
dim_patients.to_csv(PROCESSED_DIR / "dim_patients.csv", index=False)
fact_clinical_outcomes.to_csv(PROCESSED_DIR / "fact_clinical_outcomes.csv", index=False)
dim_biomarkers.to_csv(PROCESSED_DIR / "dim_biomarkers.csv", index=False)
fact_patient_biomarkers.to_csv(PROCESSED_DIR / "fact_patient_biomarkers.csv", index=False)
fact_kpis.to_csv(PROCESSED_DIR / "fact_kpis.csv", index=False)

print("CSV SQL exportés avec succès.")
print("\nFichiers créés :")
print("- dim_patients.csv")
print("- fact_clinical_outcomes.csv")
print("- dim_biomarkers.csv")
print("- fact_patient_biomarkers.csv")
print("- fact_kpis.csv")

print("\nAperçu des types :")
print("\ndim_patients")
print(dim_patients.dtypes)

print("\nfact_clinical_outcomes")
print(fact_clinical_outcomes.dtypes)

print("\ndim_biomarkers")
print(dim_biomarkers.dtypes)

print("\nfact_patient_biomarkers")
print(fact_patient_biomarkers.dtypes)

print("\nfact_kpis")
print(fact_kpis.dtypes)

CSV SQL exportés avec succès.

Fichiers créés :
- dim_patients.csv
- fact_clinical_outcomes.csv
- dim_biomarkers.csv
- fact_patient_biomarkers.csv
- fact_kpis.csv

Aperçu des types :

dim_patients
patient_id      int32
age             int64
sex            object
cancer_type    object
risk_group     object
dtype: object

fact_clinical_outcomes
patient_id           int32
survival_months      int64
event                int32
stage               object
ecog               float64
dtype: object

dim_biomarkers
biomarker_id      int32
gene_symbol      object
gene_id          object
official_name    object
description      object
dtype: object

fact_patient_biomarkers
patient_id             int32
biomarker_id           int32
association_source    object
dtype: object

fact_kpis
cancer_type         object
total_patients       int32
mean_age           float64
death_rate         float64
mean_survival      float64
median_survival    float64
dtype: object
